In [ ]:
# ==============================================================================
# 🛠️ Étape 1 : Installations et Importations des Bibliothèques
# ==============================================================================
!pip install -q wordcloud matplotlib requests spacy scikit-learn nltk
!python -m spacy download en_core_web_sm

import re
import requests
import matplotlib.pyplot as plt
import nltk
import spacy
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Téléchargements des ressources NLTK nécessaires
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

# Chargement du modèle spaCy en anglais
nlp = spacy.load("en_core_web_sm")

# ==============================================================================
# 📦 Étape 2 : Chargement et Nettoyage Intégral des Textes (Task 1 & 2)
# ==============================================================================
print("=== Étape 2 : Téléchargement et Slicing du Contenu ===")

# URLs officielles du Project Gutenberg pour les trois livres de Lewis Carroll
urls = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",      # Alice's Adventures in Wonderland
    "https://www.gutenberg.org/cache/epub/12/pg12.txt",      # Through the Looking-Glass
    "https://www.gutenberg.org/cache/epub/29042/pg29042.txt" # A Tangled Tale
]

titles = ["Alice in Wonderland", "Through the Looking-Glass", "A Tangled Tale"]

def load_and_trim_texts(urls_list):
    raw_corpus = []
    for url in urls_list:
        response = requests.get(url)
        text = response.text
        
        # Slicing intelligent pour enlever les licences de début et de fin du Project Gutenberg
        start_markers = ["*** START OF THE PROJECT GUTENBERG", "***START OF THE PROJECT GUTENBERG"]
        end_markers = ["*** END OF THE PROJECT GUTENBERG", "***END OF THE PROJECT GUTENBERG"]
        
        start_idx = 0
        end_idx = len(text)
        
        for marker in start_markers:
            if marker in text:
                start_idx = text.find(marker) + len(text[text.find(marker):text.find("\n", text.find(marker))+1])
                break
        for marker in end_markers:
            if marker in text:
                end_idx = text.find(marker)
                break
                
        trimmed_text = text[start_idx:end_idx].strip()
        raw_corpus.append(trimmed_text)
    return raw_corpus

raw_books = load_and_trim_texts(urls)

# Aperçu des 200 premiers caractères pour valider la suppression des crédits
for title, book_text in zip(titles, raw_books):
    print(f"\n📝 {title} (200 premiers caractères) :")
    print(book_text[:200], "\n" + "-"*40)

# ==============================================================================
# ⚙️ Étape 3 : Tokenisation, Stopwords et Normalisation (Task 3 à 7)
# ==============================================================================
print("\n=== Étape 3 : Traitement Textuel NLTK & spaCy ===")

# 3. Tokenisation
tokenized_books = [nltk.word_tokenize(book.lower()) for book in raw_books]
for title, tokens in zip(titles, tokenized_books):
    print(f"👉 {title} possède {len(tokens)} tokens au total. 150 premiers tokens :", tokens[:20])

# 4. Suppression des Stopwords et Ponctuation
stop_words = set(nltk.corpus.stopwords.words('english'))

cleaned_tokens_books = []
for tokens in tokenized_books:
    # On garde uniquement les chaînes alphabétiques pour filtrer la ponctuation
    cleaned = [t for t in tokens if t.isalpha() and t not in stop_words]
    cleaned_tokens_books.append(cleaned)

print("\nVérification de la suppression des mots vides ('i', 'me') dans le Livre 1 :")
print(f"Occurrence de 'i' : {cleaned_tokens_books[0].count('i')} | Occurrence de 'me' : {cleaned_tokens_books[0].count('me')}")

# 5. Racinage (Stemming) avec PorterStemmer
stemmer = nltk.PorterStemmer()
stemmed_sample = [stemmer.stem(t) for t in cleaned_tokens_books[0][:50]]
print("\n50 premiers tokens Stemmés (Livre 1) :\n", stemmed_sample)

# 6. Lemmatisation avec spaCy (Exécutée sur le texte brut pour conserver les propriétés linguistiques)
lemmatized_books = []
for book_text in raw_books:
    # Utilisation de nlp.pipe pour optimiser le traitement des longs documents
    doc = nlp(book_text[:50000]) # Tronqué à 50k caractères pour des raisons de mémoire d'exécution démo
    lemmatized_books.append([token.lemma_.lower() for token in doc if token.is_alpha and token.lemma_.lower() not in stop_words])

print("\n50 premiers tokens Lemmatisés (Livre 1) :\n", lemmatized_books[0][:50])

# ==============================================================================
# 🔍 Étape 4 : Analyse Linguistique (Task 7 à 9)
# ==============================================================================
print("\n=== Étape 4 : Analyse Linguistique Différentielle ===")
print("""
💡 Différence Stemming vs Lemmatisation :
- Le Stemming (NLTK PorterStemmer) applique des règles de découpe brutes pour tronquer la fin des mots (ex: 'travailing' -> 'travail'). Cela peut générer des non-mots qui n'existent pas dans le dictionnaire.
- La Lemmatisation (spaCy) analyse la morphologie et le contexte grammatical pour ramener le mot à sa forme canonique réelle du dictionnaire (son lemme, ex: 'went' -> 'go', 'mice' -> 'mouse').
""")

# 8. Repérage des étiquettes POS (Part-Of-Speech)
print("\nPOS Tagging (20 premiers mots du Livre 1) :")
pos_tags = nltk.pos_tag(cleaned_tokens_books[0][:20])
print(pos_tags)

# 9. Reconnaissance d'Entités Nommées (NER)
print("\nNER Extraction (Aperçu des premières entités détectées dans le Livre 1) :")
tree = nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(raw_books[0][:2000])))
entities = []
for subtree in tree:
    if hasattr(subtree, 'label'):
        entity = " ".join([token for token, pos in subtree.leaves()])
        entities.append((entity, subtree.label()))
print(entities[:10])

# ==============================================================================
# 📊 Étape 5 : Analyse Statistique, WordCloud et Bag of Words (BoW)
# ==============================================================================
print("\n=== Étape 5 : Visualisation Graphique (WordCloud & BoW) ===")

# Préparation du texte nettoyé pour les modèles (la version Lemmatisée est idéale ici)
final_corpus_strings = [" ".join(book) for book in lemmatized_books]

# 1. Génération des Word Clouds
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for i, (title, text_data) in enumerate(zip(titles, final_corpus_strings)):
    wc = WordCloud(width=400, height=400, background_color="white", max_words=100).generate(text_data)
    axes[i].imshow(wc, interpolation="bilinear")
    axes[i].set_title(title, fontsize=14, fontweight="bold")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

# 2 & 3. Bag of Words (BoW) via CountVectorizer
vectorizer_bow = CountVectorizer(max_features=1000)
bow_matrix = vectorizer_bow.fit_transform(final_corpus_strings)
feature_names = vectorizer_bow.get_feature_names_out()

print("\n--- Analyse de la Matrice Bag of Words ---")
print(f"Dimensions de la matrice BoW : {bow_matrix.shape} (3 Documents x 1000 Mots les plus fréquents)")

# Extraction et traçage des camemberts (Pie Plots) des 5 mots les plus fréquents
for doc_idx, title in enumerate(titles):
    row_data = bow_matrix.toarray()[doc_idx]
    top_5_indices = np.argsort(row_data)[::-1][:5]
    
    words = [feature_names[idx] for idx in top_5_indices]
    counts = [row_data[idx] for idx in top_5_indices]
    
    # Impression des correspondances demandées : [Numéro doc, Index, Fréquence]
    print(f"\nTop 5 pour {title} :")
    for w, c, idx in zip(words, counts, top_5_indices):
        print(f" -> Doc: {doc_idx} | Index du Mot: {idx} | Mot: '{w}' | Trouvé: {c} fois")
        
    plt.figure(figsize=(5, 5))
    plt.pie(counts, labels=words, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
    plt.title(f"BoW : Top 5 fréquences - {title}")
    plt.show()

print("\n💡 Analyse critique du BoW : Les mots extraits (comme 'alice', 'say', 'little') sont très attendus mais peu informatifs sur les spécificités de chaque histoire, car ils reviennent de manière transverse.")

# ==============================================================================
# 🎯 Étape 6 : Résolution de la Fréquence par TF-IDF
# ==============================================================================
print("\n=== Étape 6 : Correction de Fréquence par TF-IDF ===")

# Utilisation des hyperparamètres restrictifs dictés par la consigne pour notre mini-corpus
vectorizer_tfidf = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = vectorizer_tfidf.fit_transform(final_corpus_strings)
tfidf_feature_names = vectorizer_tfidf.get_feature_names_out()

# Traçage des nouveaux diagrammes basés sur l'importance TF-IDF
for doc_idx, title in enumerate(titles):
    row_data = tfidf_matrix.toarray()[doc_idx]
    top_5_indices = np.argsort(row_data)[::-1][:5]
    
    words = [tfidf_feature_names[idx] for idx in top_5_indices]
    scores = [row_data[idx] for idx in top_5_indices]
    
    plt.figure(figsize=(5, 5))
    plt.pie(scores, labels=words, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("Set2"))
    plt.title(f"TF-IDF : Top 5 Importance - {title}")
    plt.show()

print("\n✅ Analyse critique de TF-IDF : Les résultats sont nettement plus discriminants ! Les mots génériques partagés s'effacent au profit de concepts uniques propres à chaque trame narrative.")